# 05. Xe II: свой расчёт рассеяния в модели Ванга — проверка, метастабили, каскады

Три вещи, которых нет в ноутбуке 02 (там σ берутся готовыми у Ванга):

1. **Проверка σ Ванга.** Та же мишень (`models.xe2_wang`, 68 состояний, нумерация уровней —
   таблица Ванга), свой расчёт DBSR. Сравниваются σ и ⟨σv⟩ для **всех** пар, которые есть у
   Ванга (gs→6s/5d/6p, 6s→6p, 5d→6p). Это воспроизведение расчёта, а не другая физическая
   модель: совпадение проверяет, что Ванг и мы посчитали одно и то же (сетка, число
   парциальных волн, пороги). Зависимость от модели мишени — ноутбук 06.
2. **Переходы между метастабилями** (и с ²P₁/₂): у Ванга их нет. Метастабиль здесь — уровень
   ниже 6p, у которого время жизни по E1 (из этого же расчёта) длиннее `TAU_META` или
   E1-распада нет вовсе. M1 и E2 не считаются (dbsr_dmat3 падает на M1), поэтому настоящие
   времена жизни короче — проверьте по литературе, если важно.
3. **Каскады в верхние уровни выбранных линий.** Для каждого уровня h выше u — вероятность
   P(h→u), что h, распадаясь, пройдёт через u (все цепочки, ветвления DBSR E1). Константа
   каскадного заселения из уровня i: k_casc(i→u) = Σ_h k(i→h)·P(h→u). Отношение
   k_casc/k(i→u) — атомные данные (как σ_app у Fursa); как каскады войдут в заселённость, решает CRM.
   В модели Ванга выше 6p есть только 5d и 7s (до 18.5 эВ); 6d, 7p, 4f — ноутбук 06.

Все σ здесь — DBSR (этот расчёт); ветвления — DBSR E1 (этот расчёт).

**Время**: J = 0 (2 волны) на 4 ядрах — см. `docs/crm/lines_final.md`; полный расчёт
(J ≤ 25) — сервер. Сходимость по J проверьте (`JMAX` 15/20/25): у оптически разрешённых
переходов (gs→6s, 5d) большие J вносят заметный вклад.

**Выход** (папка готового расчёта `runs/xe_plus_v2`, если он есть, иначе `runs/crm/xe2_wang_full`): `sigma_xe2_wang.xlsx` (σ из начальных уровней Ванга,
формат Ванга), `rates_xe2_wang.xlsx` (k всех возбуждений), `check_vs_wang.csv`,
`xe2_levels_tau.csv`, `xe2_sigma_metastable.xlsx`, `xe2_k_metastable.xlsx`,
`xe2_cascade_P.csv`, `xe2_k_cascade.xlsx` (+ .csv).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
ROOT    = Path("/data/files_Otdel120/pydbsr_calc")    # папка установки (большой диск)
REPO    = ROOT / "pydbsr_src"                         # распакованный архив pydbsr (или клон репозитория)
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = ROOT                                        # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = ROOT / "runs" / "crm"                       # результаты расчётов (большой диск)
XE2_RUN = ROOT / "runs" / "xe_plus_v2"                # готовый расчёт e + Xe+ (модель Ванга, J <= 25)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
import shutil
JMAX, EMAX = 25, 60.0        # J: проверьте сходимость; EMAX (эВ): σ выше — хвост 1/E (как в 02)
ref = reference.read_xlsx(XLSX)
WANG_INITIAL = sorted({i for i, j in ref.sigma})         # 1, 2, 4, 5, ... 34
W = RUNS / "xe2_wang_full"
if list((XE2_RUN / "scat").glob("h.[0-9][0-9][0-9]")):
    # готовый расчёт xe_plus_v2 (examples/xe_plus_wang2019.ipynb) — та же модель, сетка энергий
    # 0.0136 эВ до 60 эВ: рассеяние не пересчитывается, считаются только A, σ и k(Te)
    W = XE2_RUN
    om = sorted((W / "compare").glob(f"omega_J{JMAX:g}_E*.npz"),           # наибольшая Emax
                key=lambda f: -float(f.name.split("_E")[1].split("_dE")[0]))
    if om and not (W / f"omega_J{JMAX:g}.npz").exists():
        (W / f"omega_J{JMAX:g}.npz").symlink_to(om[0])
    print("готовый расчёт:", W, "| парциальных волн:", len(list((W / "scat").glob("h.[0-9][0-9][0-9]"))),
          "| Ω:", (W / f"omega_J{JMAX:g}.npz").resolve())
elif (RUNS / "xe2_wang" / "target").exists() and not (W / "target").exists():
    shutil.copytree(RUNS / "xe2_wang" / "target", W / "target")      # мишень из ноутбука 01
job = nb.Job(W, "xe2_wang", levels=XLSX, stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=EMAX, de=0.0136, multipoles="E1",
             sigma_initial=",".join(map(str, WANG_INITIAL)), te=",".join(map(str, TE)))
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz", partial_waves=False)
tg = db.Target.load(W / "target")
name = {s.nist_no: s.name for s in tg.states if s.nist_no and s.name in cs.names}   # номер Ванга -> состояние
Ecm = {lv.no: lv.energy_cm for lv in ref.levels}
STN = {s.nist_no: s for s in tg.states if s.nist_no and s.name in cs.names}             # номер -> State (J, чётность)
def dbsr_sigma(i, j):
    E, s = cs.incident_energy(name[i]), cs.sigma(name[i], name[j])
    m = np.isfinite(s) & (E > 0)
    return E[m], s[m]
def thr(i, j):
    return (Ecm[j] - Ecm[i]) / nb.CM_PER_EV

TEa = np.array(TE)
def tail_of(i, j):
    """Tail of sigma above the computed grid: ln E/E for E1-allowed i -> j (as run_model.py), else 1/E."""
    try:
        a_, b_ = STN[i], STN[j]
    except (NameError, KeyError):
        return "1/E"
    e1 = a_.parity != b_.parity and abs(a_.two_j - b_.two_j) <= 2 and a_.two_j + b_.two_j > 0
    return "dipole" if e1 else "1/E"
def k_of(E, s, thr_ev, tail="1/E"):
    """Maxwellian <sigma v> (cm3/s) on the TE grid."""
    return rates.rates_on_grid(E, s, TEa, thr_ev, tail=tail)
def show_k(title, k):
    print(f"{title:28s} " + "  ".join(f"{v:9.2e}" for v in k))
print(" " * 28 + "  ".join(f"Te={t:<6g}" for t in TEa))

## 1. σ Ванга против своего расчёта

Для каждой пары: k_DBSR/k_Ванг при каждом Te (обе константы — из σ одной и той же функцией
`rates.rates_on_grid`, хвост 1/E). Сводка — медиана и доля пар в пределах ×1.5 и ×2.

In [ ]:
import csv
check = []
for (i, j), (Ew, sw) in sorted(ref.sigma.items()):
    if i in name and j in name:
        kw = k_of(Ew, sw, thr(i, j), tail_of(i, j))
        kd = k_of(*dbsr_sigma(i, j), thr(i, j), tail_of(i, j))
        check.append(dict(i=i, j=j, label=f"{ref.label(i)} -> {ref.label(j)}",
                          **{f"ratio_Te{t:g}": d / w for t, d, w in zip(TEa, kd, kw)}))
for t in TEa:
    q = np.array([c[f"ratio_Te{t:g}"] for c in check])
    q = q[np.isfinite(q) & (q > 0)]
    print(f"Te = {t:5g} эВ: {len(q)} пар, медиана DBSR/Ванг = {np.median(q):.2f}, "
          f"в пределах ×1.5: {np.mean((q > 1 / 1.5) & (q < 1.5)):.0%}, ×2: {np.mean((q > 0.5) & (q < 2)):.0%}")
with open(W / "check_vs_wang.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(check[0])); w.writeheader(); w.writerows(check)
print("->", W / "check_vs_wang.csv")
print("\nверхние уровни выбранных линий:")
for c in check:
    if c["j"] in nb.XE2_UPPERS:
        print(f"{c['i']:3d} -> {c['j']:3d}  " + "  ".join(f"{c[f'ratio_Te{t:g}']:5.2f}" for t in TEa))

In [ ]:
fig, axs = plt.subplots(2, len(nb.XE2_LINES), figsize=(4 * len(nb.XE2_LINES), 6.5), squeeze=False)
for col, x in enumerate(nb.XE2_LINES):
    u = x["upper"]
    for row, i in enumerate((1, 2)):
        ax = axs[row][col]
        if (i, u) in ref.sigma:
            E, s = ref.sigma[(i, u)]; ax.plot(E, s / 1e-16, "k", lw=0.8, label="Ванг 2019")
        E, s = dbsr_sigma(i, u); ax.plot(E, s / 1e-16, "r", lw=0.6, label=f"DBSR, J ≤ {JMAX}")
        ax.set_xscale("log"); ax.set_title(f"{i} → {u} ({x['wl']} нм)", fontsize=9)
axs[0][0].legend(fontsize=7); fig.supxlabel("E, эВ"); fig.supylabel("σ, 10⁻¹⁶ см²"); fig.tight_layout(); plt.show()

## 2. Времена жизни (E1) и метастабили

`TAU_META` — порог: уровни ниже 6p с τ(E1) длиннее него (или без E1-распада) считаются
метастабилями. Уровень 2 (²P₁/₂) распадается только M1 — он всегда в списке.

In [ ]:
TAU_META = 1e-6            # с
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")
first6p = min(n for n in name if ")6p" in ref.label(n))
low = [n for n in sorted(name) if Ecm[n] < Ecm[first6p]]
META = [n for n in low if n > 1 and (n not in tot or 1 / tot[n] > TAU_META)]
rows = []
for n in sorted(name):
    tau = 1 / tot[n] if n in tot else float("inf")
    rows.append(dict(no=n, label=ref.label(n), E_eV=Ecm[n] / nb.CM_PER_EV, tau_E1_s=tau, metastable=n in META))
    if n in low:
        print(f"{n:3d} {ref.label(n):34s} {Ecm[n] / nb.CM_PER_EV:7.3f} эВ   τ(E1) = {tau:9.3g} с" + ("   метастабиль" if n in META else ""))
with open(W / "xe2_levels_tau.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("метастабили:", META, " (начальные уровни у Ванга:", WANG_INITIAL, ")")

## 3. σ и ⟨σv⟩ между основным, ²P₁/₂ и метастабилями

Набор M — 1, 2, метастабили и начальные уровни Ванга ниже 6p (у Ванга это и 6s/5d с
τ ≈ 0.1–0.2 мкс, которые в плазме тоже заметно заселены). Только возбуждение i → j (E_i < E_j). Обратный процесс — детальное равновесие:
`rates.deexcitation(k, g_i, g_j, ΔE, Te)`, g = 2J+1 (столбец J в «NIST Level Table»).
Плюс k(i → u) из этих уровней в верхние уровни выбранных линий (у Ванга есть не все).

In [ ]:
M = sorted({1, 2} | set(META) | {i for i in WANG_INITIAL if Ecm[i] < Ecm[first6p]})   # + начальные уровни Ванга ниже 6p
sig_m, lab_m, k_m = {}, {}, {}
for a_, i in enumerate(M):
    for j in M[a_ + 1:] + [u for u in nb.XE2_UPPERS]:
        if Ecm[j] <= Ecm[i] or (i, j) in sig_m:
            continue
        E, s = dbsr_sigma(i, j)
        sig_m[(i, j)] = (E, s)
        lab_m[(i, j)] = f"{i}->{j}" + ("" if (i, j) in ref.sigma or j not in nb.XE2_UPPERS else " (нет у Ванга)")
        k_m[(i, j)] = k_of(E, s, thr(i, j), tail_of(i, j))
reference.write_xlsx(W / "xe2_sigma_metastable.xlsx", ref.levels, sig_m, sheet_of=lambda k: f"from {k[0]}", labels=lab_m)
reference.write_rates_xlsx(W / "xe2_k_metastable.xlsx", TEa, k_m, lab_m,
                           note=f"<sigma v> (cm3/s), excitation i->j, levels: Wang 2019 numbers; sigma: DBSR, model xe2_wang, J<={JMAX}")
for key in sorted(k_m):
    show_k(lab_m[key], k_m[key])

## 3б. Проверка через силы линий (gf): какому набору σ из метастабилей верить

Возбуждение m → u (m — уровень 6s/5d, u — 6p) и излучение u → m — один и тот же переход.
Для разрешённого перехода σ(m→u) примерно пропорционально его силе линии (gf ∝ A):
при больших энергиях строго, у порога — приближённо. Наш расчёт даёт gf всех таких
переходов, NIST — для части из них. Если наш A(u→m) больше NIST в X раз, то и наше
σ(m→u), вероятно, завышено примерно в X раз.

Столбцы:
* `наш/NIST` — наш A против NIST (нет в NIST — пусто);
* `наш/изм.` — по вашим спектрам: (ветвление DBSR) / (измеренное ветвление), медиана по всем
  спектрам `docs/crm/branching/` (только верхние уровни ваших линий и видимые ветви). Это
  отношение A, только если время жизни уровня в расчёте верно: BR_ист/BR_наш = (A_ист/A_наш)·(τ_ист/τ_наш).
  Если знаете измеренные времена жизни уровней 6p, впишите их в `TAU_MEAS_NS` — поправка учтётся.
  Где есть и спектр, и NIST, верим спектру (для 680.57 нм спектр противоречит NIST);
* `vel/len` — согласие двух форм нашего расчёта (далеко от 1 — линии не верим);
* `k_наш/k_Ванг` — при `TE_CHECK` из раздела 1;
* `исправл./Ванг` — k_наш × (A_истинное/A_наш) / k_Ванг: если близко к 1, расхождение с Вангом
  объясняется силой линии в нашей модели, и правее Ванг; если далеко от 1 — дело не в A.

In [ ]:
TE_CHECK = 1.0                        # эВ
lines_nist = nb.nist_lines(NIST / "nist_XeII.csv")
nos = sorted(Ecm, key=Ecm.get)        # номера уровней Ванга по возрастанию энергии
E_sorted = [Ecm[n] for n in nos]
A_nist = {}
for x in lines_nist:
    if not x["A"] or x["Ei"] is None or x["Ek"] is None:
        continue
    lo = nb.level_number(E_sorted, x["Ei"], tol=1.0)
    up = nb.level_number(E_sorted, x["Ek"], tol=1.0)
    if lo and up:
        A_nist[(nos[up - 1], nos[lo - 1])] = x["A"]
A_our, gauge = {}, {}
for r in nb.read_csv(W / "transitions_E1.csv"):
    if r["upper_no"] and r["lower_no"]:
        key = (int(r["upper_no"]), int(r["lower_no"]))
        A_our[key] = nb.fnum(r["A_exp"])
        gauge[key] = nb.fnum(r["gauge_ratio"])
# измерено по спектрам (examples/crm/branching_from_spectrum.py): доля ветви среди видимых × (1 − доля
# невидимых по DBSR, строка «ИТОГ») = измеренное ветвление BR_ист; BR_ист / BR_наш = (A_ист/A_наш)·(τ_ист/τ_наш),
# поэтому A_ист/A_наш = (BR_ист/BR_наш)·(τ_наш/τ_ист). Без измеренного τ (TAU_MEAS_NS) берётся τ_ист = τ_наш.
TAU_MEAS_NS = {}            # {номер верхнего уровня: измеренное время жизни, нс} — впишите из литературы
meas = {}
for fcsv in sorted((NIST / "branching").glob("branching_*.csv")):
    rs = nb.read_csv(fcsv)
    unseen = {r["upper"]: nb.fnum(r["unseen_dbsr"]) for r in rs if r["branch_nm"] == "ИТОГ"}
    for r in rs:
        share = nb.fnum(r["share_of_seen"])
        if not (share and r["lower"].strip().isdigit() and unseen.get(r["upper"]) is not None):
            continue
        u_, l_ = int(r["upper"]), int(r["lower"])
        bd = br.get(u_, {}).get(l_)                     # ветвление DBSR этого расчёта (раздел 2)
        if bd:
            tau_fix = (1e9 / tot[u_]) / TAU_MEAS_NS[u_] if u_ in TAU_MEAS_NS else 1.0
            meas.setdefault((u_, l_), []).append(share * (1 - unseen[r["upper"]]) / bd * tau_fix)
meas = {k: float(np.median(v)) for k, v in meas.items()}
ck = {(c["i"], c["j"]): c for c in check}
rows = []
print(f"{'m->u':>8s}  {'λ, нм':>8s}  {'A_наш':>9s}  {'A_NIST':>9s}  {'наш/NIST':>8s}  {'наш/изм.':>8s}  {'vel/len':>7s}"
      f"  {'k_наш/k_Ванг':>12s}  {'исправл./Ванг':>13s}")
for u in nb.XE2_UPPERS:
    for m in sorted(M):
        if (u, m) not in A_our or Ecm[m] >= Ecm[u]:
            continue
        a, an = A_our[(u, m)], A_nist.get((u, m))
        mr = meas.get((u, m))                          # A_истинное / A_наш по спектру
        kr = ck.get((m, u), {}).get(f"ratio_Te{TE_CHECK:g}")
        true_over_our = mr if mr else (an / a if (an and a) else None)   # спектр важнее NIST
        fix = kr * true_over_our if (kr and true_over_our) else None
        wl = 1e7 / (Ecm[u] - Ecm[m]) / 1.000277
        rows.append(dict(m=m, u=u, wl_nm=wl, A_our=a, A_NIST=an, our_NIST=a / an if an else None,
                         our_measured=1 / mr if mr else None,
                         vel_len=gauge.get((u, m)), k_our_Wang=kr, corrected_Wang=fix))
        f = lambda v, fmt: format(v, fmt) if v is not None else "—"
        print(f"{m:3d}->{u:<3d}  {wl:8.3f}  {a:9.2e}  {f(an, '9.2e'):>9s}  {f(a / an if an else None, '8.2f'):>8s}"
              f"  {f(1 / mr if mr else None, '8.2f'):>8s}"
              f"  {f(gauge.get((u, m)), '7.2f'):>7s}  {f(kr, '12.2f'):>12s}  {f(fix, '13.2f'):>13s}")
with open(W / "xe2_gf_check.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(rows[0])); wr.writeheader(); wr.writerows(rows)
print("->", W / "xe2_gf_check.csv")

## 4. Каскады в верхние уровни выбранных линий

P(h→u) — доля распадов h, проходящих через u (ветвления DBSR E1, все цепочки). Печатаются
уровни с P > `P_MIN`. k_casc(i→u) = Σ_h k(i→h)·P(h→u); отношение к прямому k(i→u).

In [ ]:
P_MIN = 1e-3
P = nb.cascade_prob(br)
prow, k_c, lab_c = [], {}, {}
kcache = {}
def k_pair(i, j):
    if (i, j) not in kcache:
        kcache[(i, j)] = k_of(*dbsr_sigma(i, j), thr(i, j), tail_of(i, j))
    return kcache[(i, j)]
for x in nb.XE2_LINES:
    u = x["upper"]
    feed = sorted(((h, P[h][u]) for h in P if P[h].get(u, 0) > P_MIN and h in name), key=lambda t: -t[1])
    print(f"\n{x['wl']} нм, верхний уровень {u} {ref.label(u)}: питающих уровней {len(feed)}")
    for h, p in feed:
        prow.append(dict(upper=u, line_nm=x["wl"], h=h, h_label=ref.label(h), P=p))
        print(f"   {h:3d} {ref.label(h):34s} P = {p:.3f}")
    for i in M:
        if Ecm[i] >= Ecm[u]:
            continue
        kc = sum(k_pair(i, h) * p for h, p in feed if Ecm[h] > Ecm[i])
        kd = k_pair(i, u)
        k_c[(i, u, "casc")] = kc; lab_c[(i, u, "casc")] = f"{i}->{u} cascade"
        k_c[(i, u, "ratio")] = kc / kd; lab_c[(i, u, "ratio")] = f"{i}->{u} cascade/direct"
        if i in (1, 2):
            show_k(f"  из {i}: k_casc/k_прям", kc / kd)
with open(W / "xe2_cascade_P.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(prow[0])); w.writeheader(); w.writerows(prow)
reference.write_rates_xlsx(W / "xe2_k_cascade.xlsx", TEa, k_c, lab_c,
                           note="cascade: sum_h <sigma v>(i->h) P(h->u), cm3/s; ratio = cascade/direct; DBSR, model xe2_wang")
print("->", W / "xe2_cascade_P.csv", W / "xe2_k_cascade.xlsx")

## 5. Набор k с поправкой по спектру (третий набор для CRM)

Для переходов m → u, у которых настоящая сила линии известна (ваш спектр — в приоритете,
иначе NIST, раздел 3б), наше k(m→u) умножается на A_истинное / A_наш. Это поправка
дипольной части сечения: точна при больших энергиях, у порога — приближённо (там вклад
обмена и резонансов не масштабируется). Для остальных пар k не меняются — берите их из
`rates_xe2_wang.xlsx`.

В CRM: три набора k(метастабиль → 6p) — Ванг, наш, наш с поправкой. Разброс ne по ним —
неопределённость от атомных данных; поправленный набор лучше всего согласован с вашим спектром.

In [ ]:
k_cor, lab_cor, cor_rows = {}, {}, []
for (u, m), a in sorted(A_our.items()):
    if m not in M or u not in nb.XE2_UPPERS or Ecm[m] >= Ecm[u] or not a:
        continue
    f_true = meas.get((u, m)) or (A_nist[(u, m)] / a if (u, m) in A_nist else None)
    if not f_true:
        continue
    src = "спектр" if (u, m) in meas else "NIST"
    k_cor[(m, u)] = f_true * k_pair(m, u)
    lab_cor[(m, u)] = f"{m}->{u}"
    cor_rows.append(dict(m=m, u=u, factor=f_true, source=src))
    show_k(f"{m}->{u} ×{f_true:.2f} ({src})", k_cor[(m, u)])
reference.write_rates_xlsx(W / "xe2_k_corrected.xlsx", TEa, k_cor, lab_cor,
                           note="<sigma v> (cm3/s), m->u: DBSR (xe_plus_v2) x A_true/A_DBSR (A_true: branching measured "
                                "from the spectra, else NIST); other pairs: rates_xe2_wang.xlsx")
with open(W / "xe2_k_corrected_factors.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(cor_rows[0])); wr.writeheader(); wr.writerows(cor_rows)
print("->", W / "xe2_k_corrected.xlsx", W / "xe2_k_corrected_factors.csv")

## 6. Поиск ne-пар, устойчивых к выбору атомных данных

Скорость заселения верхнего уровня u: K(u) = k(1→u) + w·Σ_m k(m→u), где w — доля ионов на каждом
метастабиле m относительно основного (m из `LEVER_SET`: метастабили и долгоживущие уровни, у
которых σ есть и у Ванга, и у нас). Отношение двух линий R = (BR_a K_a)/(BR_b K_b); ветвления от
ne не зависят и в производные не входят.

* **чувствительность к ne** — d ln R / d ln w: насколько меняется отношение, если доля метастабилей
  (а она растёт с ne) меняется. Важно: при Te ≈ 1 эВ из метастабилей возбуждение в 10⁵–10⁷ раз
  эффективнее, чем из основного, поэтому уже при w ≳ 10⁻⁵ оба уровня заселяются почти только из
  метастабилей, и R перестаёт зависеть от w — пара теряет чувствительность к ne. Тогда ne влияет на
  R только через **соотношение заселённостей разных метастабилей** — это решает ваша CRM, здесь
  все метастабили взяты с одной долей w. Поэтому смотрите столбцы при разных w (`W_SCAN`) и берите
  w из вашей CRM;
* **чувствительность к Te** — d ln R / d ln Te при той же w;
* **устойчивость к атомным данным** — ошибка ln ne, которую даёт замена нашего набора на
  Ванговский: |ln R_наш − ln R_Ванг| / (d ln R/d ln w);
* `ошибка ne от 10% Te` — |d ln R/d ln Te| · 0.1 / (d ln R/d ln w).

Хорошая пара: заметная чувствительность к ne в обоих наборах, малые обе «ошибки ne».

In [ ]:
TE_PAIR = 1.0                  # эВ
SN_MIN = 50
W_REF = 1e-3                   # доля ионов на каждом метастабиле относительно основного (возьмите из CRM)
W_SCAN = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
LONG_TAU = 1e-7                # с: долгоживущие уровни тоже в наборе (уровень 4: τ ≈ 220 нс)
LEVER_SET = sorted(m for m in M if m in WANG_INITIAL and m > 1 and Ecm[m] < Ecm[first6p]
                   and (m in META or (m in tot and 1 / tot[m] > LONG_TAU)))
print("метастабили в расчёте:", LEVER_SET)
TEg = np.array([TE_PAIR, 1.25 * TE_PAIR])

def k_both(i, u):
    """(k_наш, k_Ванг) на TEg или None, если у Ванга нет σ."""
    if (i, u) not in ref.sigma:
        return None
    Ew, sw = ref.sigma[(i, u)]
    kd = rates.rates_on_grid(*dbsr_sigma(i, u), TEg, thr(i, u), tail=tail_of(i, u))
    kw = rates.rates_on_grid(Ew, sw, TEg, thr(i, u), tail=tail_of(i, u))
    return kd, kw

crit = [r for r in nb.read_csv(NIST / "criteria_A3.csv")
        if r["ion"] == "XeII" and r["upper_no"] and int(r["upper_no"]) in name and ")6p" in ref.label(int(r["upper_no"]))
        and not r["nist_within_1.5fwhm"]]
def best_sn(r):
    v = [nb.fnum(r[k]) for k in r if k.endswith("_S/N") and r[k]]
    return max(v) if v else 0.0
def n_unsat(r):
    return sum(1 for k in r if k.endswith("_sat") and r[k.replace("_sat", "_S/N")] and r[k] != "SAT")
crit = [r for r in crit if best_sn(r) >= SN_MIN and n_unsat(r) > 0]
KD, KW = {}, {}                  # u -> (k1 на TEg, Σ_m k(m->u) на TEg)
KDm, KWm = {}, {}                # u -> {m: k(m->u) на TEg}
for u in sorted({int(r["upper_no"]) for r in crit}):
    k1 = k_both(1, u)
    terms = {m: k_both(m, u) for m in LEVER_SET if Ecm[m] < Ecm[u]}
    if k1 is None or any(t is None for t in terms.values()):
        continue
    KD[u] = (k1[0], sum(t[0] for t in terms.values()))
    KW[u] = (k1[1], sum(t[1] for t in terms.values()))
    KDm[u] = {m: t[0] for m, t in terms.items()}
    KWm[u] = {m: t[1] for m, t in terms.items()}
lines_c = [r for r in crit if int(r["upper_no"]) in KD]
lever = {u: dict(LD=KD[u][1][0] / KD[u][0][0], LW=KW[u][1][0] / KW[u][0][0],
                 s=float(np.log(KD[u][0][1] / KD[u][0][0]) / np.log(1.25))) for u in KD}

def lnR(K, a, b, w, it=0):
    return np.log((K[a][0][it] + w * K[a][1][it]) / (K[b][0][it] + w * K[b][1][it]))
def sens(K, a, b, w):
    """d ln R / d ln w и d ln R / d ln Te при доле w."""
    sw = (lnR(K, a, b, 2 * w) - lnR(K, a, b, w / 2)) / np.log(4)
    st = (lnR(K, a, b, w, 1) - lnR(K, a, b, w, 0)) / np.log(1.25)
    return sw, st

print(f"линий-кандидатов: {len(lines_c)}, верхних уровней: {len(KD)}")
best = {}
for ia, a in enumerate(lines_c):
    for b in lines_c[ia + 1:]:
        ua, ub = int(a["upper_no"]), int(b["upper_no"])
        if ua == ub:
            continue
        key = frozenset((ua, ub))
        dwl = abs(float(a["wl_nm"]) - float(b["wl_nm"]))
        if key not in best or dwl < best[key]["dwl"]:          # одна пара линий на пару уровней: ближайшая по λ
            best[key] = dict(line_a=float(a["wl_nm"]), up_a=ua, line_b=float(b["wl_nm"]), up_b=ub, dwl=dwl)
pairs = []
for p in best.values():
    a, b = p["up_a"], p["up_b"]
    swD, stD = sens(KD, a, b, W_REF)
    swW, stW = sens(KW, a, b, W_REF)
    s_ne = min(abs(swD), abs(swW)) if swD * swW > 0 else 0.0
    data_err = abs(lnR(KD, a, b, W_REF) - lnR(KW, a, b, W_REF)) / s_ne if s_ne > 0 else np.inf
    te_err = 0.1 * max(abs(stD), abs(stW)) / s_ne if s_ne > 0 else np.inf
    p.update(sens_ne=s_ne, sens_ne_our=swD, sens_ne_wang=swW, sens_Te_our=stD, sens_Te_wang=stW,
             ne_err_data=data_err, ne_err_10pct_Te=te_err,
             **{f"sens_ne_w{w:g}": sens(KD, a, b, w)[0] for w in W_SCAN})
    pairs.append(p)
cur = [p for p in pairs if {p["up_a"], p["up_b"]} == {41, 44}]
# чувствительность к заселённости отдельного метастабиля m (остальные — с долей W_REF):
# d ln R / d ln w_m. Если доля всех метастабилей ~ ne, R от общего уровня не зависит (см. выше),
# и ne «видно» только через изменение соотношения метастабилей — его даёт ваша CRM.
def lnR_m(Km, K1, a, b, wm, it=0):
    Ka = K1[a][0][it] + sum(wm.get(m, W_REF) * Km[a][m][it] for m in Km[a])
    Kb = K1[b][0][it] + sum(wm.get(m, W_REF) * Km[b][m][it] for m in Km[b])
    return np.log(Ka / Kb)
for p in pairs:
    a, b = p["up_a"], p["up_b"]
    sm = {}
    for m in LEVER_SET:
        if m in KDm[a] or m in KDm[b]:
            sd = (lnR_m(KDm, KD, a, b, {m: 2 * W_REF}) - lnR_m(KDm, KD, a, b, {m: W_REF / 2})) / np.log(4)
            sw_ = (lnR_m(KWm, KW, a, b, {m: 2 * W_REF}) - lnR_m(KWm, KW, a, b, {m: W_REF / 2})) / np.log(4)
            sm[m] = (sd, sw_)
    mbest = max(sm, key=lambda m: min(abs(sm[m][0]), abs(sm[m][1])) if sm[m][0] * sm[m][1] > 0 else 0.0)
    p.update(meta_best=mbest, sens_meta_our=sm[mbest][0], sens_meta_wang=sm[mbest][1],
             **{f"sens_m{m}_our": v[0] for m, v in sm.items()}, **{f"sens_m{m}_wang": v[1] for m, v in sm.items()})
print("\nчувствительность к отдельным метастабилям (лучший m для пары; d lnR / d ln w_m, наш / Ванг):")
pm = sorted((p for p in pairs if p["sens_meta_our"] * p["sens_meta_wang"] > 0),
            key=lambda p: -min(abs(p["sens_meta_our"]), abs(p["sens_meta_wang"])))
for p in pm[:15]:
    print(f"  {p['line_a']:9.3f} ({p['up_a']}) / {p['line_b']:9.3f} ({p['up_b']}):  метастабиль {p['meta_best']:3d}"
          f" {ref.label(p['meta_best']):28s} {p['sens_meta_our']:6.2f} / {p['sens_meta_wang']:6.2f}   Δλ = {p['dwl']:.1f}")
if cur:
    p = cur[0]
    print("  текущая 41/44: " + ", ".join(f"m={m}: {p.get(f'sens_m{m}_our', float('nan')):.2f}/{p.get(f'sens_m{m}_wang', float('nan')):.2f}"
                                       for m in LEVER_SET))
good = sorted((p for p in pairs if p["sens_ne"] > 0.05), key=lambda p: p["ne_err_data"] + p["ne_err_10pct_Te"])
print(f"\nпар уровней: {len(pairs)}; с чувствительностью к ne > 0.05 в обоих наборах при w = {W_REF:g}: {len(good)}")
print(f"{'линия a':>9s} {'ур.':>4s} {'линия b':>9s} {'ур.':>4s} {'d lnR/d ln w':>12s} {'d lnR/d lnTe':>12s}"
      f" {'ош.ne данные':>12s} {'ош.ne 10%Te':>11s} {'Δλ':>6s}")
for p in good[:20]:
    print(f"{p['line_a']:9.3f} {p['up_a']:4d} {p['line_b']:9.3f} {p['up_b']:4d} {p['sens_ne']:12.3f}"
          f" {p['sens_Te_our']:12.2f} {np.exp(p['ne_err_data']):11.2f}× {np.exp(p['ne_err_10pct_Te']):10.2f}× {p['dwl']:6.1f}")
if cur:
    p = cur[0]
    print("\nтекущая пара 41/44 (561.667/545.045): d lnR/d ln w (наш набор) при w = "
          + ", ".join(f"{w:g}: {p[f'sens_ne_w{w:g}']:.3f}" for w in W_SCAN))
    print("  при малой чувствительности на реальных w пара работает через соотношение метастабилей — это решает CRM")
with open(W / "xe2_ne_pairs.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(pairs[0])); wr.writeheader()
    wr.writerows(sorted(pairs, key=lambda p: -p["sens_ne"]))
print("->", W / "xe2_ne_pairs.csv")

## 7. Таблица линий для общей подгонки

Все линии-кандидаты раздела 6 (видимые, без бленд, не насыщены хотя бы в одном листе A3) с
данными, нужными CRM: ветвление (DBSR; измеренное, где есть), согласие двух форм A (vel/len),
k(1→u) и рычаг метастабилей в обоих наборах, наклон. Подгонка по всем линиям сразу:
неизвестные ne, Te (и плотность ионов как общий множитель); минимизируется сумма
[ln(I_изм/I_CRM)]² по линиям с весами 1/σ². Линии, которые систематически не ложатся, —
признак плохих атомных данных для их верхнего уровня (смотрите `vel/len` и `BR_источник`).

In [ ]:
fit_rows = []
for r in sorted(lines_c, key=lambda r: float(r["wl_nm"])):
    u, lo = int(r["upper_no"]), int(r["lower_no"]) if r["lower_no"] else None
    brd = br.get(u, {}).get(lo) if lo else None                    # ветвление DBSR (раздел 2)
    ratio = meas.get((u, lo)) if lo else None                      # A_истинное / A_наш по спектру
    g = gauge.get((u, lo)) if lo else None
    fit_rows.append(dict(wl_nm=float(r["wl_nm"]), upper=u, upper_label=ref.label(u), lower=lo,
                         lower_label=ref.label(lo) if lo else "", BR_dbsr=brd,
                         BR_best=brd * ratio if (brd and ratio) else brd,
                         BR_source="спектр" if ratio else "DBSR", vel_len=g,
                         A_reliable=(g is not None and 0.7 < g < 1.4),
                         lever_our=lever[u]["LD"], lever_wang=lever[u]["LW"], Te_slope=lever[u]["s"],
                         best_SN=best_sn(r), n_unsaturated_sheets=n_unsat(r)))
with open(W / "xe2_lines_for_fit.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(fit_rows[0])); wr.writeheader(); wr.writerows(fit_rows)
print(f"{len(fit_rows)} линий ->", W / "xe2_lines_for_fit.csv")
print("из них с надёжной A (0.7 < vel/len < 1.4):", sum(r["A_reliable"] for r in fit_rows))